In [2]:
import sys
import torch
from config import SimulationConfig, PersonalityConfig, ModelConfig
from questionbank import QuestionBank
from retriever import RetrieverModel, QuestionRetriever
from storememory import MemoryStore
from timestamp import TimestampManager
from models.teacheragent import TeacherAgent
from models.studentlearning import StudentLearningAgent
from models.studentexam import StudentExamAgent
from environments.learningloop import LearningLoop
from environments.examloop import ExamLoop
from evaluation.metrics import AnswerEvaluator
from main import run_simulation, set_seed

print("All modules imported successfully")
print(f"CUDA available: {torch.cuda.is_available()}")


All modules imported successfully
CUDA available: True


In [ ]:
#using local model
config = SimulationConfig(
    random_seed=42,
    exam_topic="Algebra"
)
config.model_config.model_type = "local"
config.learning_config.learning_rounds = 100 
config.exam_config.num_questions = 50 
config.personality = PersonalityConfig.get_high_openness()

print(f"Configuration:")
print(f"  Personality: {config.personality.name}")
print(f"  Topic: {config.exam_topic}")
print(f"  Learning rounds: {config.learning_config.learning_rounds}")
print(f"  Exam questions: {config.exam_config.num_questions}")
print(f"  Random seed: {config.random_seed}")


Using LOCAL model
Detected 2 GPU(s)
  LLM (Llama3) -> cuda:0
  Retriever -> cuda:1
Configuration:
  Personality: high_openness
  Topic: Algebra
  Learning rounds: 100
  Exam questions: 50
  Random seed: 42


In [ ]:
#using API model
config = SimulationConfig(
    random_seed=42,
    exam_topic="Geometry",# Algebra Geometry Counting & Probability Number Theory
    model_config=ModelConfig(model_type="api") 
)

config.learning_config.learning_rounds = 1 
config.exam_config.num_questions = 1


#config.personality = PersonalityConfig.get_high_openness()
#config.personality = PersonalityConfig.get_high_conscientiousness()  
#config.personality = PersonalityConfig.get_high_extraversion()   
config.personality = PersonalityConfig.get_high_agreeableness()    
#config.personality = PersonalityConfig.get_high_neuroticism()     

print(f"Configuration:")
print(f"  Model type: {config.model_config.model_type}")
print(f"  Personality: {config.personality.name}")
print(f"  Topic: {config.exam_topic}")
print(f"  Learning rounds: {config.learning_config.learning_rounds}")
print(f"  Exam questions: {config.exam_config.num_questions}")
print(f"  Random seed: {config.random_seed}")

Using API model: gpt-oss-120b
  API base URL: https://api.ai.it.ufl.edu
Configuration:
  Model type: api
  Personality: high_agreeableness
  Topic: Geometry
  Learning rounds: 1
  Exam questions: 1
  Random seed: 42


In [ ]:
results = run_simulation(config, config.personality)

print("\n=== Learning Summary ===")
print(f"Total memories: {results['learning_summary']['total_memories']}")
print(f"Action counts: {results['learning_summary']['action_counts']}")
print(f"Learning timestamp: {results['learning_summary']['final_timestamp']}")

print("\n=== Exam Summary ===")
print(f"Accuracy: {results['exam_summary']['accuracy']:.4f}")
print(f"Correct: {results['exam_summary']['correct_count']}/{results['exam_summary']['total_questions']}")
print(f"Exam timestamp: {results['exam_summary']['final_timestamp']}")

#accuracy calculation has some errors, use macro F1 as the final result
print(f"Micro F1: {results['metrics']['micro_f1']:.4f}")
print(f"Macro F1: {results['metrics']['macro_f1']:.4f}")
print(f"Micro Precision: {results['metrics']['micro_precision']:.4f}")
print(f"Micro Recall: {results['metrics']['micro_recall']:.4f}")
print(f"Exam results: output/{config.exam_topic}/agent_{config.personality.name}_lr{config.learning_config.learning_rounds}.csv")
print(f"Memory: memory/agent_{config.personality.name}_lr{config.learning_config.learning_rounds}.json")
print(f"Final results: results/result_{config.personality.name}_lr{config.learning_config.learning_rounds}.csv")